In [2]:
#import modules and load the data
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from numpy import mean
from numpy import absolute
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
import optuna

df = pd.read_csv("insurance.csv")
df.head()

c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
#Prepare the data
df = df.dropna() #remove missings
y = df['charges'] #select Target
X = df.drop('charges',axis=1) #Select features
X = pd.get_dummies(X, drop_first=True) #Turn Fuel_Type into dummy variables
columns = X.columns #create index with column names (needed for last step)
scaler = MinMaxScaler() #initiate the scaler
X = scaler.fit_transform(X) #scale the data
X = pd.DataFrame(X,columns=columns) #turn back into a dataframe

In [4]:
#Neural network with cross validation (first attempt)

#First define a function that contains the neural network model
def create_baseline():
    # create model
    nnmodel = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer 
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(1)  # Output layer
    ])
    # Compile model
    nnmodel.compile(loss='mean_absolute_error', #loss is calculated with MAE
                    optimizer='adam', #type of gradient descent
                    metrics=['mae']) #use MAE to evaluate the model
    return nnmodel

# evaluate model 

#define the estimator
estimator = KerasRegressor(build_fn=create_baseline, #you have to use the kerasRegressor wrapper for nn models
                           epochs=200, batch_size=64, #100 epochs and 64 batches
                           verbose=0) #this will shorten the output text
#Define the kfold function
kfold = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42) #I'm using just 5 splits and 1 repeat to save time, but more splits and repeats will be more accurate
#Generate the results
results = cross_val_score(estimator, X, y, cv=kfold,scoring='neg_mean_absolute_error')
#Print the results
print('the average prediction error is: %.0f' % mean(absolute(results)))

c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\MINORDDM\MINORDDM\T04_Storytelling_with_Data\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


the average prediction error is: 1608


In [5]:
def create_baseline():
    # create model
    nnmodel = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer 
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(1)  # Output layer
    ])
    nnmodel.compile(loss='mean_absolute_error', #loss is calculated with MAE
                    optimizer='adam', #type of gradient descent
                    metrics=['mae']) #use MAE to evaluate the model
    return nnmodel

final_model = create_baseline() #create the model

final_model.fit(X, y, epochs=200, batch_size=64, verbose=1) #fit the model to the data

Epoch 1/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 13236.7549 - mae: 13236.7549
Epoch 2/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10784.8115 - mae: 10784.8115
Epoch 3/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7345.0986 - mae: 7345.0986
Epoch 4/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5719.6240 - mae: 5719.6240
Epoch 5/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4378.1812 - mae: 4378.1812
Epoch 6/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3723.9797 - mae: 3723.9797
Epoch 7/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3510.9944 - mae: 3510.9944
Epoch 8/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3407.9922 - mae: 3407.9922
Epoch 9/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3253.6318 - mae: 3253.6318
Epoch 10/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3218.8770 - mae: 3218.8770
Epoch 11/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3179.6741 - mae: 3179.6741
Epoch 12/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/s

In [6]:
#Prediction of new data

#read in the data
new_patients = pd.read_csv('new_patients.csv')

#prepare the data
X_new = pd.get_dummies(new_patients,drop_first=True)

#Make sure it has the same columns as the data it was trained on
columns = X.columns
X_new = X_new.reindex(columns=columns, fill_value=0)

#Make sure it is transformed in the same way as the training data
X_new = scaler.transform(X_new) 

#make the predictions
predictions = final_model.predict(X_new)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step


In [7]:
print(predictions)

[[14195.836 ]
 [ 7667.794 ]
 [ 5897.1772]
 [13872.643 ]
 [14664.526 ]
 [ 5612.559 ]
 [12484.887 ]
 [ 4939.506 ]
 [ 6750.4365]
 [ 2713.5642]]


In [8]:
predictions = predictions.flatten()

results_table = pd.DataFrame({
    'Patient': [f'Patient {i+1}' for i in range(len(predictions))],
    'Prediction': predictions
})

print(results_table)

      Patient    Prediction
0   Patient 1  14195.835938
1   Patient 2   7667.793945
2   Patient 3   5897.177246
3   Patient 4  13872.642578
4   Patient 5  14664.526367
5   Patient 6   5612.559082
6   Patient 7  12484.886719
7   Patient 8   4939.505859
8   Patient 9   6750.436523
9  Patient 10   2713.564209
